# Architecture A-PR — Single-agent with Prompt Repetition

This notebook runs Architecture **A** (single-agent baseline) with **Prompt Repetition** enabled.

**Prompt Repetition**: Based on Leviathan et al., "Prompt Repetition Improves Non-Reasoning LLMs" (arXiv:2512.14982, 2025).
The technique repeats user prompts (`<QUERY>` → `<QUERY>\n\n<QUERY>`) to allow each token to attend to all other tokens.

Note operative:
- Imposta un token HuggingFace valido (HF Inference API) quando richiesto.
- Lancia pochi task della HumanEval per evitare costi/tempo eccessivi.
- I log sono salvati sia su stdout sia in file JSONL/LOG per analisi successive.

In [ ]:
import os
import sys
import subprocess
import pathlib

REPO_URL = "https://github.com/LLM4SE-group-15/ArchitecturesForCodeDevelopmentWithLLMs.git"
REPO_DIR = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print(f"Using repo at {REPO_DIR.resolve()}")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import getpass
from huggingface_hub import login

#placeholder
os.environ["PROMPT_REPETITION"] = "true"  # Enable prompt repetition for RQ4

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
try:
    user_info = api.whoami(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face as:", user_info.get("name") or user_info.get("user"))
except Exception as exc:
    print("Login check failed:", exc)

In [ ]:
import json
import time
import logging
import pathlib
import os

from datetime import datetime

DEFAULT_ROOT = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")
ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else pathlib.Path.cwd()

sys.path.insert(0, str(ROOT))

LOG_DIR = ROOT / "log"
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("architecture_A_PR")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "architecture_A_PR.log")
stream_handler = logging.StreamHandler()
for handler in (file_handler, stream_handler):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger ready. Repo root: %s", ROOT)
logger.info("Log files: %s", LOG_DIR)
logger.info("PROMPT_REPETITION: %s", os.environ.get("PROMPT_REPETITION", "false"))
print("Logs ->", LOG_DIR)

In [ ]:
import time
import json
import random
from src.data.task_loader import HumanEvalTaskLoader
from src.graph.graph import run_graph
from src.agents.llm import Architecture

ARCH = Architecture.A
SEED_FIXED = 31

def run_humaneval_benchmark(limit: int = 15, shuffle: bool = True):
    """
    Run benchmark on HumanEval tasks with Prompt Repetition enabled.
    
    Args:
        limit: Number of tasks to run
        shuffle: If True, randomly sample tasks with fixed seed
    """
    loader = HumanEvalTaskLoader()
    all_tasks = loader.load_all()  # Carica tutti i 164 task
    
    if shuffle:
        random.seed(SEED_FIXED)
        tasks = random.sample(all_tasks, min(limit, len(all_tasks)))
    else:
        tasks = all_tasks[:limit]

    results = []
    total = len(tasks)
    logger.info("Loaded %s tasks from HumanEval (shuffle=%s, seed=%s)", total, shuffle, SEED_FIXED)
    logger.info("Prompt Repetition: ENABLED")
    print(f"Starting benchmark on {total} tasks (Prompt Repetition: ON)...")
    
    for idx, task in enumerate(tasks, 1):
        logger.info("Running %s/%s %s", idx, total, task.task_id)
        print(f"[{idx}/{total}] Task {task.task_id} ({task.entry_point})... ", end="", flush=True)
        
        start = time.time()
        
        state = run_graph(
            task_id=task.task_id,
            task_description=task.prompt,
            test_code=task.test,
            entry_point=task.entry_point,
            architecture=ARCH,
        )

        elapsed = time.time() - start
        
        record = {
            "task_id": task.task_id,
            "entry_point": task.entry_point,
            "architecture": "A-PR",
            "prompt_repetition": True,
            "test_passed": state["test_passed"],
            "developer_tier": state.get("developer_tier"),
            "escalations": state["escalations"],
            "story_points_initial": state.get("story_points_initial"),
            "story_points_final": state.get("story_points_current"),
            "elapsed_seconds": elapsed,
        }
        results.append(record)
        
        logger.info(
            "Finished %s | pass=%s tier=%s escalations=%s elapsed=%.1fs",
            task.task_id,
            state["test_passed"],
            record["developer_tier"],
            record["escalations"],
            elapsed,
        )
        status_str = "PASS" if state["test_passed"] else "FAIL"
        print(f"{status_str} in {elapsed:.1f}s")

        with open(LOG_DIR / "architecture_A_PR.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
            
    return results

# Esecuzione: 15 task random (seed=31 per riproducibilita)
sample_results = run_humaneval_benchmark(limit=15, shuffle=True)

# Sommario
passed_count = sum(1 for r in sample_results if r['test_passed'])
print(f"\nBenchmark Completed. Passed: {passed_count}/{len(sample_results)}")

In [ ]:
!cd log && cat architecture_A_PR.jsonl

## Evaluation Metrics for Architecture A-PR

This section calculates the evaluation metrics as specified in `evaluation.md`:

- **Primary Metrics**: Pass Rate, Pass@1
- **Cost Metrics**: Execution Time, API Calls
- **Comparison**: A vs A-PR (RQ4 - Prompt Repetition effect)

In [ ]:
import json
import pandas as pd

# Load results
results_file = LOG_DIR / "architecture_A_PR.jsonl"
records = []
with open(results_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} task results")
df.head(10)

In [ ]:
# Calculate metrics
total_tasks = len(df)
passed_tasks = df['test_passed'].sum()
pass_rate = passed_tasks / total_tasks * 100
avg_time = df['elapsed_seconds'].mean()
total_time = df['elapsed_seconds'].sum()

print("=" * 50)
print("ARCHITECTURE A-PR (Single-agent + Prompt Repetition)")
print("=" * 50)
print(f"Total Tasks:     {total_tasks}")
print(f"Passed:          {passed_tasks}")
print(f"Pass Rate:       {pass_rate:.1f}%")
print(f"Avg Time/Task:   {avg_time:.2f}s")
print(f"Total Time:      {total_time:.1f}s")
print("=" * 50)
print("\nPrompt Repetition: ENABLED (arXiv:2512.14982)")